In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
# If you're in Colab, run this once, then Runtime → Restart runtime.
%pip -q install "tensorflow==2.17.0" "pandas==2.2.2" "scikit-learn==1.5.2" \
                 "opencv-python==4.10.0.84" "tqdm==4.66.4"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 0.8.3 requires tqdm>=4.67, but you have tqdm 4.66.4 which is incompatible.
tensorflow-decision-forests 1.12.0 requires tensorflow==2.19.0, but you have tensorflow 2.17.0 which is incompatible.
umap-learn 0.5.9.post2 requires scikit-learn>=1.6, but you have scikit-learn 1.5.2 which is incompatible.


In [2]:
!pip install -U numpy==1.26.4
!pip install -U pandas tqdm


  Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl.metadata (91 kB)
  Using cached tqdm-4.67.1-py3-none-any.whl.metadata (57 kB)
Using cached pandas-2.3.3-cp312-cp312-manylinux_2_24_x86_64.manylinux_2_28_x86_64.whl (12.4 MB)
Using cached tqdm-4.67.1-py3-none-any.whl (78 kB)
  Attempting uninstall: tqdm
    Found existing installation: tqdm 4.66.4
    Uninstalling tqdm-4.66.4:
      Successfully uninstalled tqdm-4.66.4
  Attempting uninstall: pandas
    Found existing installation: pandas 2.2.2
    Uninstalling pandas-2.2.2:
      Successfully uninstalled pandas-2.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.2, but you have pandas 2.3.3 which is incompatible.
cudf-cu12 25.6.0 requires pandas<2.2.4dev0,>=2.0, but you have pandas 2.3.3 which is incompatible.
dask-cudf-cu12 25.6.0 r

In [36]:
import os, math, json, random
from pathlib import Path
import numpy as np, pandas as pd, cv2
from tqdm.auto import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.applications import xception as xcep

from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score,
    precision_recall_fscore_support, confusion_matrix
)
from sklearn.model_selection import train_test_split

import gradio as gr

# ======== EDIT ME ========
IMG_DIR      = "/content/drive/MyDrive/faceforensics_benchmark_images"         # folder with 0001.png, 0002.png, ...
WEIGHTS_PATH = "/content/drive/MyDrive/xception_ffpp_weights.h5" # your Keras .h5 (binary head)
OUT_DIR      = "/content/drive/MyDrive/ffpp_autolabel_out"
TARGET_UNCERTAIN = 300     # aim to manually label ~this many (auto-tunes the band width)
# =========================

IMG_EXTS   = {".png",".jpg",".jpeg",".PNG",".JPG",".JPEG"}
IMG_SIZE   = (299, 299)
BATCH_SIZE = 64
SEED = 1234

os.makedirs(OUT_DIR, exist_ok=True)
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)

print("TF:", tf.__version__)
print("TF:", tf.__version__)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)


TF: 2.17.0
TF: 2.17.0
NumPy: 1.26.4
Pandas: 2.3.3


In [30]:
def list_images(root):
    root = Path(root)
    assert root.exists(), f"IMG_DIR not found: {root}"
    imgs = [p for p in root.rglob("*") if p.is_file() and p.suffix in IMG_EXTS]
    if not imgs:
        raise FileNotFoundError(f"No images found under {root}")
    return sorted(imgs)

imgs = list_images(IMG_DIR)
df = pd.DataFrame({"image_path": [str(p) for p in imgs]})
df["video_id"] = df["image_path"].apply(lambda s: Path(s).stem)
print(f"Found {len(df)} images. Example:\n", df.head())


Found 1000 images. Example:
                                           image_path video_id
0  /content/drive/MyDrive/faceforensics_benchmark...     0000
1  /content/drive/MyDrive/faceforensics_benchmark...     0001
2  /content/drive/MyDrive/faceforensics_benchmark...     0002
3  /content/drive/MyDrive/faceforensics_benchmark...     0003
4  /content/drive/MyDrive/faceforensics_benchmark...     0004


In [31]:
def preprocess_img(fp):
    img = tf.io.read_file(fp)
    img = tf.image.decode_image(img, channels=3, expand_animations=False)
    img = tf.image.resize(img, IMG_SIZE, method='bilinear')
    img = xcep.preprocess_input(img)  # [-1,1]
    return img

def make_ds(frame_df, batch_size=BATCH_SIZE):
    paths  = frame_df["image_path"].values
    labels = np.zeros(len(paths), dtype=np.int32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels))
    ds = ds.map(lambda p,l: (preprocess_img(p), l), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    return ds

def build_xception_binary():
    base = xcep.Xception(include_top=False, weights=None, input_shape=(299,299,3))
    x = layers.GlobalAveragePooling2D()(base.output)
    out = layers.Dense(1, activation='sigmoid')(x)
    return models.Model(base.input, out)

# load model/weights
try:
    model = tf.keras.models.load_model(WEIGHTS_PATH, compile=False)
    print("Loaded full model:", WEIGHTS_PATH)
except Exception as e:
    print("Falling back to architecture + load_weights:", e)
    model = build_xception_binary()
    model.load_weights(WEIGHTS_PATH)
    print("Loaded weights:", WEIGHTS_PATH)

# inference
ds = make_ds(df)
probs = []
for batch_imgs, _ in tqdm(ds, total=math.ceil(len(df)/BATCH_SIZE), desc="Infer"):
    pr = model.predict(batch_imgs, verbose=0).ravel()
    probs.append(pr)
probs = np.concatenate(probs, axis=0).astype(np.float32)

scores = df.copy()
scores["prob_fake"] = probs
scores.to_csv(f"{OUT_DIR}/all_scores.csv", index=False)
print("Scores saved:", f"{OUT_DIR}/all_scores.csv", "range:", float(probs.min()), "→", float(probs.max()))


Loaded full model: /content/drive/MyDrive/xception_ffpp_weights.h5


Infer:   0%|          | 0/16 [00:00<?, ?it/s]

Scores saved: /content/drive/MyDrive/ffpp_autolabel_out/all_scores.csv range: 0.4999977648258209 → 0.5000030994415283


In [32]:
def gaussian_intersection(m0, s0, m1, s1):
    a = 1/(2*s0*s0) - 1/(2*s1*s1)
    b = m1/(s1*s1) - m0/(s0*s0)
    c = (m0*m0)/(2*s0*s0) - (m1*m1)/(2*s1*s1) - np.log(s1/s0)
    if abs(a)<1e-12:
        return float(-c/b) if abs(b)>1e-12 else float((m0+m1)/2)
    disc = b*b - 4*a*c
    if disc < 0:
        return float((m0+m1)/2.0)
    x1 = (-b + np.sqrt(disc)) / (2*a)
    x2 = (-b - np.sqrt(disc)) / (2*a)
    cand = x1 if (min(m0,m1) <= x1 <= max(m0,m1)) else x2
    return float(np.clip(cand, 0.0, 1.0))

def fit_threshold(scores):
    x = scores.reshape(-1,1)
    try:
        gmm = GaussianMixture(n_components=2, random_state=SEED).fit(x)
        means = gmm.means_.ravel(); stds = np.sqrt(gmm.covariances_.ravel())
        order = np.argsort(means)
        m0, m1 = means[order[0]], means[order[1]]
        s0, s1 = stds[order[0]], stds[order[1]]
        th = gaussian_intersection(m0, s0, m1, s1)
        return th, (m0, s0, m1, s1), "gmm"
    except Exception:
        km = KMeans(n_clusters=2, n_init=10, random_state=SEED).fit(x)
        c0 = scores[km.labels_==0].mean(); c1 = scores[km.labels_==1].mean()
        th = float((c0 + c1) / 2.0)
        return th, (min(c0,c1), 0.1, max(c0,c1), 0.1), "kmeans"

probs_np = scores["prob_fake"].values.astype(np.float32)
th, (m0,s0,m1,s1), how = fit_threshold(probs_np)
print({"threshold": th, "method": how, "means": [float(m0), float(m1)]})

# choose a band around threshold to get ~TARGET_UNCERTAIN images
# find delta so that count(|score - th| <= delta) ~ target
sorted_abs = np.sort(np.abs(probs_np - th))
if len(sorted_abs) == 0:
    delta = 0.05
else:
    k = np.clip(TARGET_UNCERTAIN, 1, len(sorted_abs))
    delta = float(sorted_abs[int(k)-1])
delta = max(delta, 0.02)  # minimum 0.02 band
print("Uncertainty band width (±delta):", delta)

is_uncertain = np.abs(probs_np - th) <= delta
n_unc = int(is_uncertain.sum())
print("Uncertain images:", n_unc, "of", len(scores))

# auto-label confident tails
auto = scores.copy()
auto["label"] = -1
auto.loc[probs_np >= th + delta, "label"] = 1  # confident fake
auto.loc[probs_np <= th - delta, "label"] = 0  # confident real

auto_labeled = auto[auto["label"]!=-1].copy().reset_index(drop=True)
uncertain   = auto[auto["label"]==-1].copy().reset_index(drop=True)

auto_labeled.to_csv(f"{OUT_DIR}/auto_labeled.csv", index=False)
uncertain.to_csv(f"{OUT_DIR}/uncertain_tolabel.csv", index=False)

print("Auto-labeled:", len(auto_labeled), " → real(0) =", int((auto_labeled.label==0).sum()),
      " fake(1) =", int((auto_labeled.label==1).sum()))
print("To label manually:", len(uncertain))


{'threshold': 0.4583333333333333, 'method': 'gmm', 'means': [0.5000002512931824, 0.5000002512931827]}
Uncertainty band width (±delta): 0.041666507720947266
Uncertain images: 303 of 1000
Auto-labeled: 705  → real(0) = 0  fake(1) = 705
To label manually: 295


In [34]:
OUT_CSV = f"{OUT_DIR}/labels_images.csv"  # final merged labels will be written here

# resume: start from existing CSV if present
existing = {}
if Path(OUT_CSV).exists():
    df_old = pd.read_csv(OUT_CSV)
    if {"image_path","label"}.issubset(df_old.columns):
        existing = {row.image_path: int(row.label) for _,row in df_old.iterrows()}

to_label_paths = [p for p in uncertain["image_path"].tolist() if p not in existing]

state = {
    "ordered": to_label_paths,
    "i": 0,
    "labels": existing,  # image_path -> 0/1, prefilled with auto-labeled below after UI ends
    "history": []
}

def status_line():
    done = len(state["labels"])
    total = len(auto_labeled) + len(to_label_paths)
    return f"Labeled (including auto): {done}/{total} ({done/total*100:.1f}%)"

def load_image(idx):
    if len(state["ordered"]) == 0:
        return None, f"Nothing to label. {status_line()}", "All uncertain images labeled or none existed."
    idx = max(0, min(idx, len(state["ordered"]) - 1))
    state["i"] = idx
    img_path = state["ordered"][idx]
    meta = f"{status_line()} | {Path(img_path).name} ({idx+1}/{len(state['ordered'])})"
    hint = "Mark: REAL (0) or FAKE (1). Skip to move on."
    return img_path, meta, hint

def _assign(label):
    if not state["ordered"]:
        return None, status_line(), "No images queued."
    img_path = state["ordered"][state["i"]]
    prev = state["labels"].get(img_path, None)
    state["labels"][img_path] = label
    state["history"].append((img_path, prev))
    return next_image()

def mark_real(): return _assign(0)
def mark_fake(): return _assign(1)

def skip_image():
    return next_image()

def next_image():
    if not state["ordered"]:
        return None, status_line(), "Done."
    i = min(state["i"] + 1, len(state["ordered"]) - 1)
    return load_image(i)

def prev_image():
    if not state["ordered"]:
        return None, status_line(), "Done."
    i = max(state["i"] - 1, 0)
    return load_image(i)

def undo():
    if not state["history"]:
        return load_image(state["i"])
    img_path, prev = state["history"].pop()
    if prev is None:
        state["labels"].pop(img_path, None)
    else:
        state["labels"][img_path] = prev
    try:
        idx = state["ordered"].index(img_path)
    except ValueError:
        idx = state["i"]
    return load_image(idx)

def save_csv():
    # merge: auto-labeled + manually labeled (uncertain)
    auto_rows = auto_labeled[["image_path","label"]].to_dict("records")
    manual_rows = [{"image_path": k, "label": v} for k,v in state["labels"].items()]
    merged = {}
    for r in auto_rows: merged[r["image_path"]] = r["label"]
    for r in manual_rows: merged[r["image_path"]] = r["label"]
    df_out = pd.DataFrame([{"image_path": k, "label": v} for k,v in merged.items()]).sort_values("image_path")
    df_out.to_csv(OUT_CSV, index=False)
    return f"Saved {len(df_out)} total labels to {OUT_CSV}. You can close the UI."

with gr.Blocks(title="Label Uncertain Images (0=Real, 1=Fake)") as demo:
    gr.Markdown("## Label Uncertain Images — 0=Real, 1=Fake")
    with gr.Row():
        img = gr.Image(label="Image", interactive=False)
        with gr.Column():
            meta = gr.Markdown()
            hint = gr.Markdown()
            btn_prev = gr.Button("⟵ Prev")
            btn_next = gr.Button("Next ⟶")
            btn_undo = gr.Button("Undo")
            btn_real = gr.Button("Mark REAL (0)", variant="primary")
            btn_fake = gr.Button("Mark FAKE (1)", variant="primary")
            btn_skip = gr.Button("Skip")
            btn_save = gr.Button("Save CSV")
            saved_msg = gr.Markdown()

    btn_real.click(mark_real, outputs=[img, meta, hint])
    btn_fake.click(mark_fake, outputs=[img, meta, hint])
    btn_skip.click(skip_image, outputs=[img, meta, hint])
    btn_prev.click(prev_image, outputs=[img, meta, hint])
    btn_next.click(next_image, outputs=[img, meta, hint])
    btn_undo.click(undo, outputs=[img, meta, hint])
    btn_save.click(save_csv, outputs=[saved_msg])

    demo.load(lambda: load_image(0), outputs=[img, meta, hint])

print("Launching UI. Label only the uncertain images, then click 'Save CSV'.")
demo.launch(share=False)


Launching UI. Label only the uncertain images, then click 'Save CSV'.
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

In [35]:
final_csv = Path(f"{OUT_DIR}/labels_images.csv")
if final_csv.exists():
    df_lab = pd.read_csv(final_csv)
    print("Final labels:", final_csv, "rows:", len(df_lab))
    print(df_lab["label"].value_counts().to_dict())
    # You can now set LABELS_CSV to this file in your baseline notebook:
    # LABELS_CSV = f"{OUT_DIR}/labels_images.csv"
    # and run the fully supervised metrics + F1-optimal threshold.
else:
    print("labels_images.csv not found yet — click 'Save CSV' in the UI.")


Final labels: /content/drive/MyDrive/ffpp_autolabel_out/labels_images.csv rows: 1000
{1: 859, 0: 141}
